In [1]:
#!/usr/bin/env python3
"""
Calcolo dell'indice di attrattività comunale e generazione mappa.

Questo script:
1. Carica i dati OSM aggregati per comune
2. Calcola l'area di ogni comune dalla geometria
3. Normalizza i conteggi per superficie (densità per km²)
4. Applica normalizzazione min-max (0-100) per ogni pilastro
5. Calcola l'indice ponderato composito
6. Classifica i comuni in quintili (classe 1-5)
7. Genera una mappa coropletica interattiva in HTML

Dipendenze: pip install pandas geopandas folium

Uso:
    python 02_calcolo_indice.py

Output:
    - indice_attrattivita.csv: dati completi con indice e classe
    - mappa_attrattivita_sardegna.html: mappa interattiva
"""


"\nCalcolo dell'indice di attrattività comunale e generazione mappa.\n\nQuesto script:\n1. Carica i dati OSM aggregati per comune\n2. Calcola l'area di ogni comune dalla geometria\n3. Normalizza i conteggi per superficie (densità per km²)\n4. Applica normalizzazione min-max (0-100) per ogni pilastro\n5. Calcola l'indice ponderato composito\n6. Classifica i comuni in quintili (classe 1-5)\n7. Genera una mappa coropletica interattiva in HTML\n\nDipendenze: pip install pandas geopandas folium\n\nUso:\n    python 02_calcolo_indice.py\n\nOutput:\n    - indice_attrattivita.csv: dati completi con indice e classe\n    - mappa_attrattivita_sardegna.html: mappa interattiva\n"

In [2]:
import pandas as pd
import geopandas as gpd
import folium
from folium import MacroElement
from branca.element import Element
from pathlib import Path
import numpy as np

In [3]:
# ============================================================
# CONFIGURAZIONE
# ============================================================

# Directory dei file di input/output
OUTPUT_DIR = "/home/davide/Scaricati/sardegna-overtourism-aida26-main/output_attrattivita"

# File di input
GEOJSON_FILE = f"{OUTPUT_DIR}/comuni_sardegna.geojson"
CSV_FILE = f"{OUTPUT_DIR}/osm_per_comune.csv"

# File di output
INDICE_CSV = f"{OUTPUT_DIR}/indice_attrattivita.csv"
MAPPA_HTML = f"{OUTPUT_DIR}/mappa_attrattivita_sardegna.html"

In [4]:
# ============================================================
# CONFIGURAZIONE PESI
# ============================================================
# Pesi per il calcolo dell'indice composito.
# La somma deve essere 1.0 (o 100%).

WEIGHTS = {
    "turismo": 0.30,        # 30% - Turismo e patrimonio culturale
    "natura": 0.25,         # 25% - Natura e ambiente
    "ristorazione": 0.20,   # 20% - Ristorazione e commercio
    "servizi": 0.15,        # 15% - Servizi essenziali
    "infrastrutture": 0.10, # 10% - Infrastrutture di trasporto
}

In [5]:
# Pilastri da includere nel calcolo (devono corrispondere alle colonne del CSV)
PILLARS = list(WEIGHTS.keys())

In [6]:
# (chiesto all'ai) CRS proiettato per calcolo aree (UTM zone 32N per la Sardegna)
PROJECTED_CRS = "EPSG:32632"

# Coordinate centro mappa (Sardegna)
MAP_CENTER = [40.12, 9.01]
MAP_ZOOM = 8

In [7]:
# ============================================================
# FUNZIONI DI UTILITÀ
# ============================================================

def load_data(geojson_path: str, csv_path: str) -> tuple:
    """
    Carica i dati OSM e i confini comunali.
    
    Args:
        geojson_path: Percorso al file GeoJSON dei comuni
        csv_path: Percorso al file CSV con i conteggi OSM
        
    Returns:
        Tupla (GeoDataFrame comuni, DataFrame conteggi)
    """
    print("[1/7] Caricamento dati...")
    
    comuni_gdf = gpd.read_file(geojson_path)
    counts_df = pd.read_csv(csv_path)
    
    print(f"  → {len(comuni_gdf)} comuni caricati")
    print(f"  → {len(counts_df)} righe nel CSV")
    
    return comuni_gdf, counts_df


In [29]:
def calculate_areas(comuni_gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Calcola l'area di ogni comune in km² usando un CRS proiettato.
    
    Args:
        comuni_gdf: GeoDataFrame con le geometrie comunali
        
    Returns:
        GeoDataFrame con nuova colonna 'area_km2'
    """
    print("[2/7] Calcolo aree comunali...")
    
    # Riproietta in CRS proiettato per calcolo aree accurate
    comuni_projected = comuni_gdf.to_crs(crs=PROJECTED_CRS)
    
    # Calcola area in km² (da m²)
    comuni_gdf = comuni_gdf.copy()
    comuni_gdf["area_km2"] = comuni_projected.geometry.area / 1_000_000
    
    # Calcola statistiche
    min_area = comuni_gdf["area_km2"].min()
    max_area = comuni_gdf["area_km2"].max()
    mean_area = comuni_gdf["area_km2"].mean()
    
    print(f"  → Area minima: {min_area:.2f} km²")
    print(f"  → Area massima: {max_area:.2f} km²")
    print(f"  → Area media: {mean_area:.2f} km²")
    
    return comuni_gdf

In [19]:
def normalize_by_area(counts_df: pd.DataFrame, comuni_gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    """
    Normalizza i conteggi POI per superficie comunale (densità per km²).
    
    Args:
        counts_df: DataFrame con i conteggi POI per comune
        comuni_gdf: GeoDataFrame con le aree comunali
        
    Returns:
        DataFrame con nuove colonne di densità
    """
    print("[3/7] Normalizzazione per superficie...")
    
    # Merge per ottenere le aree
    result_df = counts_df.copy()
    
    # Assicurati che codice_istat sia lo stesso tipo
    comuni_areas = comuni_gdf[["com_istat_code", "area_km2"]].copy()
    comuni_areas["com_istat_code"] = comuni_areas["com_istat_code"].astype(int)
    
    result_df = result_df.merge(
        comuni_areas, 
        left_on="codice_istat", 
        right_on="com_istat_code", 
        how="left"
    )
    
    # Calcola densità per ogni pilastro
    for pillar in PILLARS:
        if pillar in result_df.columns:
            # Densità = count / area (valori piccoli, es. 0.5 servizi/km²)
            result_df[f"{pillar}_density"] = result_df[pillar] / result_df["area_km2"]
            result_df[f"{pillar}_log_density"] = np.log(result_df[pillar] / result_df["area_km2"] + 1)
        else:
            result_df[f"{pillar}_density"] = 0
            result_df[f"{pillar}_log_density"] = 0
    
    return result_df

In [20]:
def min_max_normalize(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """
    Applica normalizzazione min-max per scalare i valori in range 0-100.
    
    Args:
        df: DataFrame con i valori da normalizzare
        columns: Lista delle colonne da normalizzare
        
    Returns:
        DataFrame con nuove colonne normalizzate
    """
    result_df = df.copy()
    
    for col in columns:
        if col in result_df.columns:
            min_val = result_df[col].min()
            max_val = result_df[col].max()
            
            if max_val > min_val:
                # Min-max normalization: (x - min) / (max - min) * 100
                result_df[f"{col}_norm"] = (
                    (result_df[col] - min_val) / (max_val - min_val) * 100
                )
            else:
                result_df[f"{col}_norm"] = 0
            
            print(f"  → {col}: min={min_val:.4f}, max={max_val:.4f}")
        else:
            result_df[f"{col}_norm"] = 0
    
    return result_df

In [21]:
def calculate_weighted_index(df: pd.DataFrame, weights: dict) -> pd.DataFrame:
    """
    Calcola l'indice composito ponderato.
    
    Args:
        df: DataFrame con le colonne normalizzate
        weights: Dizionario {pilastro: peso} (somma = 1.0)
        
    Returns:
        DataFrame con nuova colonna 'indice'
    """
    print("[4/7] Calcolo indice ponderato...")
    
    result_df = df.copy()
    
    # Calcola l'indice come somma ponderata dei pilastri normalizzati
    result_df["indice"] = 0.0
    result_df["indice_log"] = 0.0
    
    for pillar, weight in weights.items():
        norm_col = f"{pillar}_density_norm"
        norm_log_col = f"{pillar}_log_density_norm"
        if (norm_col in result_df.columns and norm_log_col in result_df.columns):
            result_df["indice"] += result_df[norm_col] * weight
            result_df["indice_log"] += result_df[norm_log_col] * weight

        print(f"  → {pillar}: peso {weight*100:.0f}%")
    
    print(f"  → Indice range: {result_df['indice'].min():.2f} - {result_df['indice'].max():.2f}")
    print(f"  → Indice range log: {result_df['indice_log'].min():.2f} - {result_df['indice_log'].max():.2f}")
    
    return result_df

In [22]:
def classify_quintiles(df: pd.DataFrame, column: str = "indice") -> pd.DataFrame:
    """
    Classifica i comuni in quintili (5 classi di attrattività).
    
    Args:
        df: DataFrame con l'indice
        column: Colonna da usare per la classificazione
        
    Returns:
        DataFrame con nuova colonna 'classe' (1-5)
    """
    print("[5/7] Classificazione in quintili...")
    
    result_df = df.copy()
    
    # Calcola i quintili (5 classi)
    # qcut assegna etichette 1-5 basate sui percentili
    result_df["classe"] = pd.qcut(
        result_df[column], 
        q=5, 
        labels=[1, 2, 3, 4, 5]
    ).astype(int)
    
    # Statistiche per classe
    for classe in range(1, 6):
        subset = result_df[result_df["classe"] == classe]
        print(f"  → Classe {classe}: {len(subset)} comuni "
              f"(indice {subset[column].min():.2f} - {subset[column].max():.2f})")
    
    return result_df

In [ ]:
#non utilizzata
""" 
def save_results(df: pd.DataFrame, output_path: str) -> None:
    
    Salva il DataFrame risultato in CSV.
    
    Args:
        df: DataFrame con i risultati
        output_path: Percorso del file di output
    
    print(f"[6/7] Salvataggio risultati in {output_path}...")
    
    # Seleziona le colonne rilevanti per il CSV
    output_df = df[[
        "codice_istat", 
        "nome_comune", 
        "area_km2",
        "turismo", "natura", "ristorazione", "servizi", "infrastrutture",
        "indice", 
        "classe"
    ]].copy()
    
    # Arrotonda i valori numerici
    output_df["area_km2"] = output_df["area_km2"].round(2)
    output_df["indice"] = output_df["indice"].round(2)
    
    output_df.to_csv(output_path, index=False, encoding="utf-8")
    print(f"  → {len(output_df)} righe salvate")
"""

In [24]:
def generate_map(comuni_gdf: gpd.GeoDataFrame, result_df: pd.DataFrame, output_path: str) -> None:
    """
    Genera una mappa coropletica interattiva con folium.
    
    Args:
        comuni_gdf: GeoDataFrame con le geometrie comunali
        result_df: DataFrame con indice e classe
        output_path: Percorso del file HTML di output
    """
    print("[7/7] Generazione mappa coropletica...")
    
    # Prepara i dati per la mappa
    comuni_for_map = comuni_gdf.merge(
        result_df[["codice_istat", "indice", "classe"] + 
                  [f"{p}_norm" for p in PILLARS]],
        left_on="com_istat_code",
        right_on="codice_istat",
        how="left"
    )
    
    # Crea la mappa base
    m = folium.Map(
        location=MAP_CENTER,
        zoom_start=MAP_ZOOM,
        tiles="CartoDB positron"
    )
    
    # Costruisci il GeoJSON inline per folium
    geojson_data = comuni_for_map.to_json()
    
    # Crea la mappa coropletica
    folium.Choropleth(
        geo_data=geojson_data,
        name="Attrattività",
        data=comuni_for_map,
        columns=["com_istat_code", "indice"],
        key_on="feature.properties.com_istat_code",
        fill_color="YlOrRd",
        fill_opacity=0.7,
        line_opacity=0.2,
        line_weight=1,
        legend_name="Indice di Attrattività",
        nan_fill_color="white",
        nan_fill_opacity=0.3
    ).add_to(m)
    
    # Aggiungi tooltip interattivo
    style_function = lambda x: {
        "fillColor": "#ffffff",
        "color": "#000000",
        "fillOpacity": 0,
        "weight": 0
    }
    
    highlight_function = lambda x: {
        "fillColor": "#000000",
        "color": "#000000",
        "fillOpacity": 0.1,
        "weight": 1
    }
    
    # Tooltip con informazioni dettagliate
    tooltip_cols = [
        "name", "indice", "classe",
        "turismo_norm", "natura_norm", "ristorazione_norm",
        "servizi_norm", "infrastrutture_norm"
    ]
    
    # Rinomina per il tooltip
    comuni_for_map["nome_comune"] = comuni_for_map["name"]
    
    # Crea il layer GeoJson con tooltip
    folium.GeoJson(
        geojson_data,
        name="Dettagli comune",
        style_function=style_function,
        highlight_function=highlight_function,
        tooltip=folium.GeoJsonTooltip(
            fields=["nome_comune", "indice", "classe",
                    "turismo_norm", "natura_norm", "ristorazione_norm",
                    "servizi_norm", "infrastrutture_norm"],
            aliases=["Comune:", "Indice:", "Classe (1-5):",
                    "Turismo:", "Natura:", "Ristorazione:",
                    "Servizi:", "Infrastrutture:"],
            localize=True,
            sticky=True,
            labels=True,
            style="""
                background-color: white;
                border: 2px solid black;
                border-radius: 3px;
                box-shadow: 3px 3px 3px rgba(0,0,0,0.3);
                font-size: 12px;
                padding: 10px;
            """
        )
    ).add_to(m)
    
    # Aggiungi pannello info HTML custom (top-right)
    info_html = """
    <div style="position: fixed; 
                top: 10px; right: 10px; 
                width: 220px;
                background-color: white; 
                border: 2px solid grey; 
                border-radius: 5px;
                padding: 10px;
                z-index: 9999;
                font-size: 12px;
                box-shadow: 3px 3px 3px rgba(0,0,0,0.3);">
        <h4 style="margin: 0 0 10px 0; border-bottom: 1px solid grey; padding-bottom: 5px;">
            Indice di Attrattività
        </h4>
        <p style="margin: 5px 0;">
            <b>5 pilastri tematici</b><br>
            Turismo, Natura, Ristorazione,<br>
            Servizi, Infrastrutture
        </p>
        <p style="margin: 5px 0;">
            <b>Classi di attrattività:</b><br>
            1 = Molto bassa<br>
            2 = Bassa<br>
            3 = Media<br>
            4 = Alta<br>
            5 = Molto alta
        </p>
        <p style="margin: 5px 0; font-size: 10px; color: #666;">
            Passa il mouse sui comuni per i dettagli
        </p>
    </div>
    """
    m.get_root().html.add_child(Element(info_html))
    
    # Aggiungi legenda custom (bottom-left)
    legend_html = """
    <div style="position: fixed; 
                bottom: 50px; left: 10px; 
                width: 150px;
                background-color: white; 
                border: 2px solid grey; 
                border-radius: 5px;
                padding: 10px;
                z-index: 9999;
                font-size: 11px;
                box-shadow: 3px 3px 3px rgba(0,0,0,0.3);">
        <h4 style="margin: 0 0 10px 0;">Legenda</h4>
        <div style="background: linear-gradient(to right, #FFFFB2, #FED976, #FEB24C, #FD8D3C, #FC4E2A, #E31A1C, #B10026); 
                    height: 20px; 
                    border-radius: 3px;
                    margin-bottom: 5px;"></div>
        <div style="display: flex; justify-content: space-between; font-size: 10px;">
            <span>Basso</span>
            <span>Alto</span>
        </div>
    </div>
    """
    m.get_root().html.add_child(Element(legend_html))
    
    # Aggiungi controllo layer
    folium.LayerControl().add_to(m)
    
    # Salva la mappa
    m.save(output_path)
    print(f"  → Mappa salvata: {output_path}")

# ============================================================
# MAIN
# ============================================================


In [ ]:
"""
def main():
    
    print("=" * 60)
    print("CALCOLO INDICE DI ATTRATTIVITÀ COMUNALE")
    print("=" * 60)
    print()
    
    # Verifica che i file di input esistano
    geojson_path = Path(GEOJSON_FILE)
    csv_path = Path(CSV_FILE)
    
    if not geojson_path.exists():
        print(f"ERRORE: File non trovato: {GEOJSON_FILE}")
        print("Esegui prima: python 01_raccolta_dati_osm.py")
        return
    
    if not csv_path.exists():
        print(f"ERRORE: File non trovato: {CSV_FILE}")
        print("Esegui prima: python 01_raccolta_dati_osm.py")
        return
    
    # 1. Carica i dati
    comuni_gdf, counts_df = load_data(GEOJSON_FILE, CSV_FILE)
    
    # 2. Calcola le aree
    comuni_gdf = calculate_areas(comuni_gdf)
    
    # 3. Normalizza per superficie
    result_df = normalize_by_area(counts_df, comuni_gdf)
    
    # 4. Applica min-max normalization per ogni pilastro
    density_cols = [f"{p}_density" for p in PILLARS]
    result_df = min_max_normalize(result_df, density_cols)
    
    # 5. Calcola l'indice ponderato
    result_df = calculate_weighted_index(result_df, WEIGHTS)
    
    # 6. Classifica in quintili
    result_df = classify_quintiles(result_df, "indice")
    
    # 7. Salva i risultati
    save_results(result_df, INDICE_CSV)
    
    # 8. Genera la mappa
    generate_map(comuni_gdf, result_df, MAPPA_HTML)
    
    print("\n" + "=" * 60)
    print("CALCOLO INDICE COMPLETATO")
    print("=" * 60)
    print(f"\nFile generati:")
    print(f"  - {INDICE_CSV}")
    print(f"  - {MAPPA_HTML}")
    
    # Mostra i top 10 comuni
    print("\n" + "-" * 40)
    print("TOP 10 COMUNI PER INDICE DI ATTRATTIVITÀ")
    print("-" * 40)
    top10 = result_df.nlargest(10, "indice")[["nome_comune", "indice", "classe"]]
    for i, (_, row) in enumerate(top10.iterrows(), 1):
        print(f"{i:2}. {row['nome_comune']:<25} {row['indice']:>6.2f} (classe {row['classe']})")


if __name__ == "__main__":
    main()

""""

In [25]:
# Verifica che i file di input esistano
geojson_path = Path(GEOJSON_FILE)
csv_path = Path(CSV_FILE)

if not geojson_path.exists():
    print(f"ERRORE: File non trovato: {GEOJSON_FILE}")
    print("Esegui prima: python 01_raccolta_dati_osm.py")

if not csv_path.exists():
    print(f"ERRORE: File non trovato: {CSV_FILE}")
    print("Esegui prima: python 01_raccolta_dati_osm.py")

In [26]:
# 1. Carica i dati
comuni_gdf, counts_df = load_data(GEOJSON_FILE, CSV_FILE)

[1/7] Caricamento dati...
  → 377 comuni caricati
  → 377 righe nel CSV


In [30]:
# 2. Calcola le aree
comuni_gdf = calculate_areas(comuni_gdf)

[2/7] Calcolo aree comunali...
  → Area minima: 3.01 km²
  → Area massima: 546.66 km²
  → Area media: 63.91 km²


In [31]:
# 3. Normalizza per superficie
result_df = normalize_by_area(counts_df, comuni_gdf)

[3/7] Normalizzazione per superficie...


In [32]:
# 4. Applica min-max normalization per ogni pilastro
density_cols = [f"{p}_density" for p in PILLARS]
density_log_cols = [f"{p}_log_density" for p in PILLARS]
result_df = min_max_normalize(result_df, density_cols)
result_df = min_max_normalize(result_df, density_log_cols)


  → turismo_density: min=0.0256, max=8.2723
  → natura_density: min=0.0000, max=13.4737
  → ristorazione_density: min=0.0000, max=6.6651
  → servizi_density: min=0.0000, max=3.5015
  → infrastrutture_density: min=0.0000, max=8.2208
  → turismo_log_density: min=0.0253, max=2.2270
  → natura_log_density: min=0.0000, max=2.6723
  → ristorazione_log_density: min=0.0000, max=2.0367
  → servizi_log_density: min=0.0000, max=1.5044
  → infrastrutture_log_density: min=0.0000, max=2.2215


In [115]:
result_df.head()

,codice_istat,nome_comune,codice_istat_str,infrastrutture,natura,ristorazione,servizi,turismo,com_istat_code,area_km2,...,ristorazione_density_norm,servizi_density_norm,infrastrutture_density_norm,turismo_log_density_norm,natura_log_density_norm,ristorazione_log_density_norm,servizi_log_density_norm,infrastrutture_log_density_norm,indice,indice_log
0,90001,Aggius,90001,0,19,14,6,12,90001,86.366311,...,2.432085,1.984078,0.000000,4.759654,7.440879,7.376188,4.464547,0.000000,1.604436,5.433036
1,90002,Alà dei Sardi,90002,2,62,16,5,40,90002,197.226348,...,1.217167,0.724031,0.123353,7.237807,10.228862,3.829882,1.664156,0.454186,1.592222,5.789576
2,90003,Alghero,90003,241,283,174,59,251,90003,223.061281,...,11.703617,7.554048,13.142526,33.091560,30.655249,28.313027,15.599431,32.976940,11.142385,28.891494
3,90004,Anela,90004,1,186,1,3,6,90004,36.734292,...,0.408435,2.332392,0.331142,5.722096,67.441701,1.318747,5.218286,1.209050,10.360534,19.744452
4,90005,Ardara,90005,2,14,3,4,8,90005,37.975382,...,1.185261,3.008221,0.640640,7.533270,11.743736,3.733211,6.656821,2.310448,2.109508,7.172125


In [33]:
# 5. Calcola l'indice ponderato
result_df = calculate_weighted_index(result_df, WEIGHTS)

[4/7] Calcolo indice ponderato...
  → turismo: peso 30%
  → natura: peso 25%
  → ristorazione: peso 20%
  → servizi: peso 15%
  → infrastrutture: peso 10%
  → Indice range: 0.61 - 77.23
  → Indice range log: 1.89 - 87.51


In [34]:
result_df

,codice_istat,nome_comune,codice_istat_str,infrastrutture,natura,ristorazione,servizi,turismo,com_istat_code,area_km2,...,ristorazione_density_norm,servizi_density_norm,infrastrutture_density_norm,turismo_log_density_norm,natura_log_density_norm,ristorazione_log_density_norm,servizi_log_density_norm,infrastrutture_log_density_norm,indice,indice_log
0,90001,Aggius,90001,0,19,14,6,12,90001,86.366311,...,2.432085,1.984078,0.000000,4.759654,7.440879,7.376188,4.464547,0.000000,1.604436,5.433036
1,90002,Alà dei Sardi,90002,2,62,16,5,40,90002,197.226348,...,1.217167,0.724031,0.123353,7.237807,10.228862,3.829882,1.664156,0.454186,1.592222,5.789576
2,90003,Alghero,90003,241,283,174,59,251,90003,223.061281,...,11.703617,7.554048,13.142526,33.091560,30.655249,28.313027,15.599431,32.976940,11.142385,28.891494
3,90004,Anela,90004,1,186,1,3,6,90004,36.734292,...,0.408435,2.332392,0.331142,5.722096,67.441701,1.318747,5.218286,1.209050,10.360534,19.744452
4,90005,Ardara,90005,2,14,3,4,8,90005,37.975382,...,1.185261,3.008221,0.640640,7.533270,11.743736,3.733211,6.656821,2.310448,2.109508,7.172125
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372,111103,Villaputzu,111103,5,102,18,6,56,111103,180.725984,...,1.494331,0.948162,0.336539,11.110170,16.745506,4.661763,2.170981,1.228488,2.555942,8.900276
373,111104,Villasalto,111104,1,46,1,4,10,111104,130.393340,...,0.115064,0.876106,0.093289,2.206732,11.306992,0.375114,2.008462,0.343911,1.004080,3.899451
374,111105,Villasimius,111105,4,124,68,7,65,111105,57.561611,...,17.724378,3.473098,0.845304,33.176399,42.986331,38.294820,7.628564,3.024252,12.162145,29.805176
375,111106,Villasor,111106,7,252,6,6,19,111106,86.406930,...,1.041832,1.983146,0.985452,7.878116,51.085689,3.296261,4.462517,3.506595,6.722401,16.814146


In [35]:
# 6. Classifica in quintili
result_df = classify_quintiles(result_df, "indice") 

[5/7] Classificazione in quintili...
  → Classe 1: 76 comuni (indice 0.61 - 1.68)
  → Classe 2: 75 comuni (indice 1.69 - 2.56)
  → Classe 3: 75 comuni (indice 2.57 - 3.81)
  → Classe 4: 75 comuni (indice 3.84 - 5.71)
  → Classe 5: 76 comuni (indice 5.72 - 77.23)


In [36]:
result_df.to_csv(f"{OUTPUT_DIR}/prova_indice.csv")